# METAR Temperature Visualization with Lonboard
---

## Overview
   
Within this notebook, we will create an interactive visualization of the latest METAR data across all stations. We will use the following libraries for our visualizations:

1. [Geopandas](https://geopandas.org/en/stable/)
2. [Lonboard](https://developmentseed.org/lonboard/latest/)

## Prerequisites
| Concepts | Importance | Notes |
| --- | --- | --- |
| [Pandas](https://foundations.projectpythia.org/core/pandas.html) | Required | Tabular Datasets |

- **Time to learn**: 10 minutes
---

In [1]:
import duckdb
import numpy as np
import pandas as pd
import geopandas as gpd
from datetime import datetime, timedelta, timezone
from lonboard import viz, Map, ScatterplotLayer, HeatmapLayer
import hvplot
import hvplot.pandas
import pgeocode
import holoviews as hv
from palettable.colorbrewer.diverging import BrBG_10

In [2]:
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import shapely
from ipywidgets import FloatRangeSlider, jsdlink

from lonboard import Map, ScatterplotLayer
from lonboard.colormap import apply_continuous_cmap
from lonboard.controls import MultiRangeSlider
from lonboard.layer_extension import DataFilterExtension

In [3]:
url = 'https://data.source.coop/dynamical/asos-parquet/year=2026/data.parquet'

## Get the past three days of data
This query will request just the latest hours worth of data across all stations.

In [4]:
time_1 = datetime.now()
print(f"end time: {time_1}")

end time: 2026-06-17 17:21:51.999182


In [5]:
time_0 = datetime.now() - timedelta(hours=72)
print(f"start time: {time_0}")

start time: 2026-06-14 17:21:52.004022


# Query the database for all data between those timestamps

In [8]:
df = duckdb.execute("""
    SELECT *
    FROM read_parquet($1, hive_partitioning=true)
    WHERE 
---      country = 'US' AND
    valid BETWEEN $2 AND $3
    ORDER BY country
""", [url, time_0, time_1]).fetchdf()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Print the first couple columns of data to get an understanding of the structure

In [11]:
df.head(4)

,station,valid,longitude,latitude,tmpf,tmpc,dwpf,dwpc,relh,drct,...,state,geometry,name,elevation,country,county,wfo,tzname,bbox,year
0,ANYN,2026-06-15 08:00:00+00:00,166.9191,-0.5475,80.6,27.0,78.8,26.0,94.29,250.0,...,AU,"[1, 1, 0, 0, 0, 29, 56, 103, 68, 105, 221, 100...",Nauru,7.0,AU,None,None,Pacific/Nauru,"{'xmin': 166.9191, 'ymin': -0.5475, 'xmax': 16...",2026
1,ANYN,2026-06-16 08:00:00+00:00,166.9191,-0.5475,80.6,27.0,75.2,24.0,83.70,240.0,...,AU,"[1, 1, 0, 0, 0, 29, 56, 103, 68, 105, 221, 100...",Nauru,7.0,AU,None,None,Pacific/Nauru,"{'xmin': 166.9191, 'ymin': -0.5475, 'xmax': 16...",2026
2,NZQN,2026-06-14 17:30:00+00:00,168.7392,-45.0211,51.8,11.0,44.6,7.0,76.34,220.0,...,AU,"[1, 1, 0, 0, 0, 129, 38, 194, 134, 167, 23, 10...",Queenstown,356.0,AU,None,None,Pacific/Auckland,"{'xmin': 168.7392, 'ymin': -45.0211, 'xmax': 1...",2026
3,NZQN,2026-06-14 18:00:00+00:00,168.7392,-45.0211,50.0,10.0,42.8,6.0,76.17,220.0,...,AU,"[1, 1, 0, 0, 0, 129, 38, 194, 134, 167, 23, 10...",Queenstown,356.0,AU,None,None,Pacific/Auckland,"{'xmin': 168.7392, 'ymin': -45.0211, 'xmax': 1...",2026


## Understanding the data columns

The documentation can be found here for the individual columns:

https://github.com/dynamical-org/asos-parquet#key-fields

Print the columns as follows:

In [13]:
df.columns

Index(['station', 'valid', 'longitude', 'latitude', 'tmpf', 'tmpc', 'dwpf',
       'dwpc', 'relh', 'drct', 'sknt', 'gust', 'alti', 'mslp', 'vsby', 'p01i',
       'p01m', 'state', 'geometry', 'name', 'elevation', 'country', 'county',
       'wfo', 'tzname', 'bbox', 'year'],
      dtype='object')

Now create bind descriptions to each of the columns

In [15]:
variables = {
    'tmpf': "Temperature in Fahrenheit",
    'tmpc': "Temperature in Celsius",
    'dwpf': "Dewpoint in Fahrenheit",
    'dwpc': "Dewpoint in Celsius", 
    'relh': "Relative humidity in percent", 
    'drct': "Wind direction in degrees", 
    'sknt': "Wind speed in knots", 
    'gust': "Wind gusts in knots", 
    'alti': "Altimeter setting in inches of mercury", 
    'mslp': "Mean sea level pressure", 
    'vsby': "Visibility in statute miles", 
    'p01i': "P01I", 
    'p01m': "P01M"
}

## Plot Timeseries data from a station local to Boulder, Colorado
Select all the data from the Boulder station, "BDU" and plot it.

In [25]:
df_bdu = df.loc[df['station'] == "BDU"]
df_bdu.head(2)

,station,valid,longitude,latitude,tmpf,tmpc,dwpf,dwpc,relh,drct,...,state,geometry,name,elevation,country,county,wfo,tzname,bbox,year
148175,BDU,2026-06-14 17:35:00+00:00,-105.2258,40.0394,59.0,15.0,42.8,6.0,54.86,350.0,...,CO,"[1, 1, 0, 0, 0, 245, 219, 215, 129, 115, 78, 9...",Boulder,1612.0,US,Boulder,BOU,America/Denver,"{'xmin': -105.2258, 'ymin': 40.0394, 'xmax': -...",2026
148176,BDU,2026-06-14 17:55:00+00:00,-105.2258,40.0394,59.0,15.0,42.8,6.0,54.86,10.0,...,CO,"[1, 1, 0, 0, 0, 245, 219, 215, 129, 115, 78, 9...",Boulder,1612.0,US,Boulder,BOU,America/Denver,"{'xmin': -105.2258, 'ymin': 40.0394, 'xmax': -...",2026


In [21]:
df_select = pd.DataFrame({
    'date': df_bdu['valid'],
    'tmpc': df_bdu['tmpc'],
    'dwpc': df_bdu['dwpc'],
    'relh': df_bdu['relh'],
    'drct': df_bdu['drct'],
    'sknt': df_bdu['sknt'],
    'alti': df_bdu['alti'],
}).set_index('date')

# print the head of the dataframe
df_select.head(2)

,tmpc,dwpc,relh,drct,sknt,alti
date,,,,,,
2026-06-14 17:35:00+00:00,15.0,6.0,54.86,350.0,4.0,30.3
2026-06-14 17:55:00+00:00,15.0,6.0,54.86,10.0,6.0,30.3


Iterate through some variables and compose an interactive plot of the past three days

In [23]:
hv.Layout(
    [df_select[i].hvplot.line(title=f"{variables[i]}", width=400) for i in [
        'tmpc',
        'dwpc',
        'relh',
        'drct',
        'sknt',
        'alti'
    ]]
).cols(3)

:Layout
   .Curve.Tmpc :Curve   [date]   (tmpc)
   .Curve.Dwpc :Curve   [date]   (dwpc)
   .Curve.Relh :Curve   [date]   (relh)
   .Curve.Drct :Curve   [date]   (drct)
   .Curve.Sknt :Curve   [date]   (sknt)
   .Curve.Alti :Curve   [date]   (alti)

In [30]:
# data_variable = 'relh'
# dates = df_select['valid']
# values = df_select[data_variable]
# df = pd.DataFrame({'date': dates, 'value': values}).set_index('date')
# df.hvplot.line(title=f"Time-Series Line Plot {data_variable}")

## Plotting Across All Stations using Lonboard

Now we will use the library (Lonboard)[https://developmentseed.org/lonboard/latest/] to plot the latest temperature data across all stations.

In [44]:
url = "https://ookla-open-data.s3.us-west-2.amazonaws.com/parquet/performance/type=mobile/year=2019/quarter=1/2019-01-01_performance_mobile_tiles.parquet"
local_path = Path("data-filter-extension.parquet")
if local_path.exists():
    gdf = gpd.read_parquet(local_path)
else:
    columns = ["avg_d_kbps", "avg_u_kbps", "avg_lat_ms", "devices", "tile"]
    df = pd.read_parquet(url, columns=columns)

    tile_geometries = shapely.from_wkt(df["tile"])
    tile_centroids = shapely.centroid(tile_geometries)
    non_geom_columns = [col for col in columns if col != "tile"]
    gdf = gpd.GeoDataFrame(
        df[non_geom_columns],
        geometry=tile_centroids,
        crs="EPSG:4326",
    )
    gdf.to_parquet(local_path)

In [32]:
gdf

,avg_d_kbps,avg_u_kbps,avg_lat_ms,devices,geometry
0,5983,7886,68,1,POINT (-160.01862 70.63722)
1,3748,5841,78,2,POINT (-160.04059 70.63357)
2,3364,6200,78,2,POINT (-160.04059 70.63175)
3,2381,2328,86,1,POINT (-160.0351 70.63357)
4,3047,5356,75,1,POINT (-160.0351 70.63175)
...,...,...,...,...,...
3231240,19528,3200,68,1,POINT (169.81842 -46.29571)
3231241,15693,10359,56,1,POINT (169.81293 -46.3071)
3231242,26747,9674,58,2,POINT (169.66461 -46.42082)
3231243,67995,13564,63,1,POINT (169.65912 -46.4511)


In [33]:
filter_extension = DataFilterExtension(filter_size=3)

In [34]:
min_bound = 5000
max_bound = 50000
normalized_download_speed = (gdf["avg_d_kbps"] - min_bound) / (max_bound - min_bound)
fill_color = apply_continuous_cmap(normalized_download_speed, BrBG_10)
radius = normalized_download_speed * 1000

In [35]:
filter_values = np.column_stack(
    [gdf["avg_d_kbps"], gdf["avg_u_kbps"], gdf["avg_lat_ms"]],
)
initial_filter_range = [
    [10_000, 50_000],
    [1000, 10_000],
    [0, 100],
]

In [42]:
#sidecar = Sidecar(anchor="split-right")
layer = ScatterplotLayer.from_geopandas(
    gdf,
    extensions=[filter_extension],
    get_fill_color=fill_color,
    get_radius=radius,
    get_filter_value=filter_values,
    filter_range=initial_filter_range,
    radius_units="meters",
    radius_min_pixels=0.1,
)
m = Map(layer)
download_slider = FloatRangeSlider(
    value=initial_filter_range[0],
    min=0,
    max=70_000,
    step=0.1,
    description="Download: ",
)
upload_slider = FloatRangeSlider(
    value=initial_filter_range[1],
    min=0,
    max=50_000,
    step=1,
    description="Upload: ",
)
latency_slider = FloatRangeSlider(
    value=initial_filter_range[2],
    min=0,
    max=500,
    step=1,
    description="Latency: ",
)
multi_slider = MultiRangeSlider([download_slider, upload_slider, latency_slider])

display(m)

In [43]:
# Important: call jsdlink to link the widgets
jsdlink((multi_slider, "value"), (layer, "filter_range"))

multi_slider

MultiRangeSlider(children=(FloatRangeSlider(value=(10000.0, 50000.0), description='Download: ', max=70000.0), …

## References

1. [GeoPandas](https://geopandas.org)
1. [EPSG](https://epsg.io)

## What's next?
Expanding on the plotting capability to more interactively visualize the wind 'u' and 'v' components.